In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder,MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix,precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import VarianceThreshold
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout,BatchNormalization
)



In [2]:
data_train=pd.read_csv('KDDTrain+.txt', header=None)
data_test=pd.read_csv('KDDTest+.txt', header=None)

In [3]:
columns = (['duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent','hot'
,'num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root','num_file_creations'
,'num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count','srv_count','serror_rate'
,'srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count'
,'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate'
,'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate','outcome','level'])
data_train.columns = columns
data_test.columns = columns

In [4]:
attack_mapping = {
    'normal': 'normal',
    'neptune': 'DoS',
    'smurf': 'DoS',
    'back': 'DoS',
    'teardrop': 'DoS',
    'pod': 'DoS',
    'land': 'DoS',
    'apache2': 'DoS',
    'mailbomb': 'DoS',
    'processtable': 'DoS',
    'udpstorm': 'DoS',
    'nuked': 'DoS',
    'worm': 'DoS',

    'ipsweep': 'Probe',
    'portsweep': 'Probe',
    'nmap': 'Probe',
    'satan': 'Probe',
    'mscan': 'Probe',
    'saint': 'Probe',
    'xsnoop': 'Probe',
    'snmpgetattack': 'Probe',
    'snmpguess': 'Probe',
    'httptunnel': 'Probe',

    'warezclient': 'R2L',
    'guess_passwd': 'R2L',
    'ftp_write': 'R2L',
    'multihop': 'R2L',
    'imap': 'R2L',
    'warezmaster': 'R2L',
    'phf': 'R2L',
    'spy': 'R2L',
    'sendmail': 'R2L',
    'secrect': 'R2L',

    'rootkit': 'U2R',
    'buffer_overflow': 'U2R',
    'loadmodule': 'U2R',
    'perl': 'U2R',
    'ps': 'U2R',
    'sqlattack': 'U2R',
    'xterm': 'U2R',
    'named': 'U2R',
    'xlock': 'U2R'
}

unmapped_attacks = set(data_train['outcome'].unique()) - set(attack_mapping.keys())
if unmapped_attacks:
    print(f"Warning: The following attack types in 'outcome' are not in the provided mapping: {unmapped_attacks}. They will be mapped to 'Other_Attack'.")
    for attack in unmapped_attacks:
        attack_mapping[attack] = 'Other_Attack'

data_train['attack_class'] = data_train['outcome'].map(attack_mapping)
data_test['attack_class'] = data_test['outcome'].map(attack_mapping)

print("\nValue counts for 'attack_class' after mapping:")
print(data_train['attack_class'].value_counts())
print("\nValue counts for 'attack_class' in test set after mapping:")
print(data_test['attack_class'].value_counts())


Value counts for 'attack_class' after mapping:
attack_class
normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64

Value counts for 'attack_class' in test set after mapping:
attack_class
normal    9711
DoS       7460
Probe     3067
R2L       2213
U2R         93
Name: count, dtype: int64


In [5]:
data_train = pd.get_dummies(
    data_train,
    columns=['protocol_type', 'service', 'flag'],
    drop_first=True
)
data_test = pd.get_dummies(
    data_test,
    columns=['protocol_type', 'service', 'flag'],
    drop_first=True
)

In [6]:
data_train, data_test = data_train.align(
    data_test,
    join='left',
    axis=1,
    fill_value=0
)

In [7]:
le=LabelEncoder()
data_train['attack_class'] = le.fit_transform(data_train['attack_class'])
data_test['attack_class'] = le.transform(data_test['attack_class'])

In [8]:
dict(zip(
    data_train['attack_class'].value_counts().index,
    data_train['attack_class'].value_counts().values
))

{4: np.int64(67343),
 0: np.int64(45927),
 1: np.int64(11656),
 2: np.int64(995),
 3: np.int64(52)}

In [9]:
X_train = data_train.drop(['attack_class', 'outcome'], axis=1)
y_train = data_train['attack_class']
X_test = data_test.drop(['attack_class', 'outcome'], axis=1)
y_test = data_test['attack_class']

In [10]:
smote = SMOTE(

     sampling_strategy={
         2: 5000,    
         3: 1500     
    },

    random_state=42,
    k_neighbors=3
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [11]:
# scaler = MinMaxScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

In [12]:
%pip install catboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier
)

from sklearn.tree import DecisionTreeClassifier

from catboost import CatBoostClassifier

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
models = {
    "XGBoost": XGBClassifier(

        objective='multi:softprob',

        num_class=5,

        n_estimators=600,

        max_depth=12,

        learning_rate=0.03,

        subsample=0.85,

        colsample_bytree=0.85,

        min_child_weight=1,

        gamma=0.1,

        reg_alpha=0.05,

        reg_lambda=1.5,

        eval_metric='mlogloss',

        tree_method='hist',

        grow_policy='lossguide',

        random_state=42
    ),
    "Random Forest": RandomForestClassifier(

        n_estimators=500,

        max_depth=30,

        min_samples_split=2,

        min_samples_leaf=1,

        max_features='sqrt',

        class_weight='balanced_subsample',

        bootstrap=True,

        random_state=42,

        n_jobs=-1
    ),
    "Decision Tree": DecisionTreeClassifier(

        criterion='gini',

        max_depth=30,

        min_samples_split=5,

        min_samples_leaf=2,

        max_features='sqrt',

        class_weight='balanced',

        random_state=42
    ),
    "AdaBoost": AdaBoostClassifier(

        n_estimators=400,

        learning_rate=0.03,

        random_state=42
    ),
    "CatBoost": CatBoostClassifier(

        iterations=700,

        depth=10,

        learning_rate=0.03,

        loss_function='MultiClass',

        eval_metric='MultiClass',

        l2_leaf_reg=3,

        bagging_temperature=1,

        random_strength=1,

        border_count=128,

        verbose=0,

        random_state=42
    ),
    "LightGBM": LGBMClassifier(

        objective='multiclass',

        num_class=5,

        n_estimators=700,

        learning_rate=0.03,

        max_depth=12,

        num_leaves=128,

        subsample=0.85,

        colsample_bytree=0.85,

        min_child_samples=20,

        reg_alpha=0.1,

        reg_lambda=1,

        random_state=42
    )
}


In [16]:
results = []
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(
        X_train_smote,
        y_train_smote
    )
    pred = model.predict(
        X_test
    )
    acc = accuracy_score(
        y_test,
        pred
    )
    precision = precision_score(
        y_test,
        pred,
        average='weighted'
    )
    recall = recall_score(
        y_test,
        pred,
        average='weighted'
    )
    f1 = f1_score(
        y_test,
        pred,
        average='weighted'
    )
    macro_f1 = f1_score(
        y_test,
        pred,
        average='macro'
    )
    results.append([
        name,
        acc,
        precision,
        recall,
        f1,
        macro_f1
    ])
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"Macro F1 : {macro_f1:.4f}")
    print("\nClassification Report:\n")
    print(
            classification_report(
                y_test,
                pred,
                target_names=[
                    str(x)
                    for x in le.classes_
                ]
            )
        )
    results_df = pd.DataFrame(
    results,
    columns=[
        'Model',
        'Accuracy',
        'Precision',
        'Recall',
        'F1 Score',
        'Macro F1'
    ]
    )
print("FINAL MODEL COMPARISON")
print(
    results_df.sort_values(
        by='Macro F1',
        ascending=False
    )
)


Training XGBoost...
Accuracy : 0.8149
Precision: 0.8418
Recall   : 0.8149
F1 Score : 0.8036
Macro F1 : 0.6715

Classification Report:

              precision    recall  f1-score   support

         DoS       0.96      0.85      0.90      7460
       Probe       0.84      0.59      0.69      3067
         R2L       0.94      0.35      0.51      2213
         U2R       0.39      0.45      0.42        93
      normal       0.73      0.97      0.83      9711

    accuracy                           0.81     22544
   macro avg       0.77      0.64      0.67     22544
weighted avg       0.84      0.81      0.80     22544

Training Random Forest...
Accuracy : 0.7482
Precision: 0.8075
Recall   : 0.7482
F1 Score : 0.7147
Macro F1 : 0.5618

Classification Report:

              precision    recall  f1-score   support

         DoS       0.96      0.77      0.86      7460
       Probe       0.86      0.48      0.62      3067
         R2L       0.93      0.05      0.10      2213
         U2R      

d:\IDS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\IDS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\IDS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\IDS\.venv\Lib\site-packages\sklearn\metrics\_classification.p

Accuracy : 0.7049
Precision: 0.6905
Recall   : 0.7049
F1 Score : 0.6761
Macro F1 : 0.4614

Classification Report:

              precision    recall  f1-score   support

         DoS       0.76      0.66      0.70      7460
       Probe       0.56      0.28      0.38      3067
         R2L       0.61      0.31      0.41      2213
         U2R       0.00      0.00      0.00        93
      normal       0.70      0.97      0.82      9711

    accuracy                           0.70     22544
   macro avg       0.53      0.44      0.46     22544
weighted avg       0.69      0.70      0.68     22544

Training CatBoost...
Accuracy : 0.8044
Precision: 0.8265
Recall   : 0.8044
F1 Score : 0.7935
Macro F1 : 0.6690

Classification Report:

              precision    recall  f1-score   support

         DoS       0.96      0.81      0.88      7460
       Probe       0.80      0.59      0.68      3067
         R2L       0.86      0.36      0.51      2213
         U2R       0.45      0.44      0.45

In [17]:
xgb=models["XGBoost"]
catb=models["CatBoost"]

In [18]:
xgb_train_probs = xgb.predict_proba(X_train)
catb_train_probs = catb.predict_proba(X_train)

xgb_test_probs = xgb.predict_proba(X_test)
catb_test_probs = catb.predict_proba(X_test)

In [19]:
X_meta_train = np.hstack([
    xgb_train_probs,
    catb_train_probs
])

X_meta_test = np.hstack([
    xgb_test_probs,
    catb_test_probs
])

print(X_meta_train.shape)

(125973, 10)


In [20]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=100,
    random_state=42
)

mlp.fit(X_meta_train, y_train)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",100
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42


In [21]:
y_pred = mlp.predict(X_meta_test)

In [22]:
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Macro F1 Score: {f1_score(y_test, y_pred, average='macro'):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.96      0.84      0.90      7460
           1       0.85      0.58      0.69      3067
           2       0.93      0.35      0.51      2213
           3       0.46      0.39      0.42        93
           4       0.73      0.97      0.83      9711

    accuracy                           0.81     22544
   macro avg       0.79      0.63      0.67     22544
weighted avg       0.84      0.81      0.80     22544

Accuracy: 0.8128
Precision: 0.8406
Recall: 0.8128
F1 Score: 0.8011
Macro F1 Score: 0.6692
Confusion Matrix:
[[6298   51    2    2 1107]
 [ 165 1785   19    8 1090]
 [   0   51  768   33 1361]
 [   0   12   30   36   15]
 [  68  203    4    0 9436]]
